# ⚡ Visual Backtest Report
### Institutional-Grade Strategy Performance Analysis

---

> *Transforming raw rolling-window backtest data into a narrative of risk and reward.*

**How to use:** Run all cells in order. When prompted, enter the index of the run you want to analyse (press **Enter** to accept the most-recent default).

In [33]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import glob, os, warnings
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── Colour palette ───────────────────────────────────────────────────────────
NAVY    = '#0D1B2A'
GREEN   = '#2ECC71'
CRIMSON = '#E74C3C'
GOLD    = '#F39C12'
SLATE   = '#5D6D7E'
LIGHT   = '#ECF0F1'
STRAT_PALETTE = ['#2ECC71', '#3498DB', '#F39C12', '#9B59B6']

# ── Modular entry-point ──────────────────────────────────────────────────────
def generate_report(dataframe):
    """Swap in any DataFrame with *_CAGR and *_MDD column pairs."""
    global df, cagr_cols, mdd_cols, strat_names
    df = dataframe.copy()
    df = df.apply(pd.to_numeric, errors='coerce')
    # Replace inf with NaN then drop rows that are entirely NaN
    df.replace([float('inf'), float('-inf')], float('nan'), inplace=True)
    n_bad = df.isnull().sum().sum()
    if n_bad:
        print(f'⚠️  Removing {n_bad} non-numeric / infinite values before plotting.')
        df.dropna(inplace=True)
    cagr_cols   = [c for c in df.columns if 'CAGR' in c]
    mdd_cols    = [c for c in df.columns if 'MDD'  in c]
    strat_names = [c.replace(' CAGR', '') for c in cagr_cols]
    if not cagr_cols:
        raise ValueError("No CAGR columns found.  Expected columns containing 'CAGR'.")
    print(f'✅ generate_report: {len(cagr_cols)} strategies detected.')
    return df

# ── Data loading ─────────────────────────────────────────────────────────────
all_runs = glob.glob('data/BT_*_*/rolling_*_results.csv')
all_runs.sort(key=os.path.getmtime, reverse=True)

if not all_runs:
    print('❌ No backtest runs found in data/BT_*_*/ directories.')
else:
    print('📋 Available Backtest Runs (Newest First):')
    for i, run in enumerate(all_runs):
        print(f'  [{i:>2}]  {os.path.dirname(run)}')
    try:
        idx = int(input('\nSelect run index [default 0]: ').strip() or 0)
        selected_run = all_runs[idx]
    except Exception:
        selected_run = all_runs[0]

    latest_folder = os.path.dirname(selected_run)
    _raw = pd.read_csv(selected_run, index_col=0, parse_dates=True)
    df   = generate_report(_raw)
    print(f'\n✅ Activated  : {latest_folder}')
    print(f'📅 Date range : {df.index.min().date()} → {df.index.max().date()}')
    print(f'📊 Windows    : {len(df):,} rolling observations')

📋 Available Backtest Runs (Newest First):
  [ 0]  data/BT_20y_1980-01-01_to_2026-04-28_20260513_072006
  [ 1]  data/BT_20y_1960-01-01_to_2026-04-28_20260512_183100
  [ 2]  data/BT_20y_1960-01-01_to_2026-04-28_20260512_182836
  [ 3]  data/BT_20y_1960-01-01_to_2026-04-28_20260512_182701
  [ 4]  data/BT_20y_1960-01-01_to_2026-04-28_20260512_181342
  [ 5]  data/BT_30y_1987-01-01_to_2026-04-28_20260512_180743
  [ 6]  data/BT_30y_1987-01-01_to_2026-04-28_20260503_201020
  [ 7]  data/BT_20y_1987-01-01_to_2024_20260503_135147
  [ 8]  data/BT_20y_1987-01-01_to_2024_20260429_211958
  [ 9]  data/BT_25y_1970-01-01_to_2024_20260429_190432
  [10]  data/BT_35y_1970-01-01_to_2024_20260429_190222
  [11]  data/BT_20y_1970-01-01_to_2024_20260429_184155
  [12]  data/BT_20y_1970-01-01_to_2024_20260429_184005
  [13]  data/BT_10y_1990-01-01_to_2024_20260428_220044
  [14]  data/BT_10y_1990-01-01_to_2024_20260428_215539
  [15]  data/BT_20y_1990-01-01_to_2024_20260428_215328
  [16]  data/BT_20y_1980-01-01_to_20

In [34]:
# ── Hero Metric Banner ───────────────────────────────────────────────────────
PRIMARY = cagr_cols[0]
P_NAME  = strat_names[0]

median_cagr  = df[PRIMARY].median() * 100
p25_cagr     = df[PRIMARY].quantile(0.25) * 100
p75_cagr     = df[PRIMARY].quantile(0.75) * 100
worst_mdd    = df[mdd_cols[0]].min() * 100
sharpe_proxy = df[PRIMARY].mean() / df[PRIMARY].std()

# Per-strategy detail rows
rows_html = ''
for i, (col, mdd_col, name) in enumerate(zip(cagr_cols, mdd_cols, strat_names)):
    c    = round(df[col].median() * 100, 2)
    c25  = round(df[col].quantile(0.25) * 100, 2)
    c75  = round(df[col].quantile(0.75) * 100, 2)
    m    = round(df[mdd_col].min()  * 100, 2)
    sp   = round(df[col].mean() / df[col].std(), 2)
    c_c  = GREEN if c >= 0 else CRIMSON
    rows_html += (
        f'<tr style="border-top:1px solid #1E2F3E;">'
        f'<td style="color:{LIGHT};padding:10px 16px;font-size:13px;">{name}</td>'
        f'<td style="color:{c_c};text-align:center;font-weight:700;">{c:+.2f}%</td>'
        f'<td style="color:{SLATE};text-align:center;font-size:12px;">{c25:+.2f}% – {c75:+.2f}%</td>'
        f'<td style="color:{CRIMSON};text-align:center;font-weight:700;">{m:.2f}%</td>'
        f'<td style="color:{GOLD};text-align:center;font-weight:700;">{sp:.2f}</td>'
        f'</tr>'
    )

banner = f"""
<div style="background:{NAVY};border-radius:14px;padding:32px 36px;
            font-family:'Segoe UI',Arial,sans-serif;margin:12px 0;">
  <h2 style="color:{LIGHT};text-align:center;margin:0 0 28px;
             letter-spacing:3px;font-size:14px;font-weight:400;">
    STRATEGY PERFORMANCE SUMMARY
  </h2>
  <table style="width:100%;border-collapse:collapse;margin-bottom:28px;">
    <tr>
      <td style="text-align:center;padding:24px;border-right:1px solid #1E2F3E;">
        <div style="color:{SLATE};font-size:11px;letter-spacing:2px;margin-bottom:10px;">MEDIAN CAGR</div>
        <div style="color:{GREEN};font-size:52px;font-weight:900;line-height:1;">{median_cagr:+.2f}%</div>
        <div style="color:{LIGHT};font-size:12px;margin-top:8px;">{P_NAME}</div>
        <div style="color:{SLATE};font-size:11px;margin-top:10px;letter-spacing:1px;">EXPECTED RANGE (P25–P75)</div>
        <div style="color:{LIGHT};font-size:16px;font-weight:700;margin-top:4px;">{p25_cagr:+.2f}% – {p75_cagr:+.2f}%</div>
      </td>
      <td style="text-align:center;padding:24px;border-right:1px solid #1E2F3E;">
        <div style="color:{SLATE};font-size:11px;letter-spacing:2px;margin-bottom:10px;">WORST MAX DRAWDOWN</div>
        <div style="color:{CRIMSON};font-size:52px;font-weight:900;line-height:1;">{worst_mdd:.1f}%</div>
        <div style="color:{LIGHT};font-size:12px;margin-top:8px;">Largest peak-to-trough loss</div>
      </td>
      <td style="text-align:center;padding:24px;">
        <div style="color:{SLATE};font-size:11px;letter-spacing:2px;margin-bottom:10px;">SHARPE PROXY</div>
        <div style="color:{GOLD};font-size:52px;font-weight:900;line-height:1;">{sharpe_proxy:.2f}</div>
        <div style="color:{LIGHT};font-size:12px;margin-top:8px;">Mean CAGR ÷ Std Dev of CAGR</div>
      </td>
    </tr>
  </table>
  <table style="width:100%;border-collapse:collapse;">
    <tr style="background:#1A2B3C;">
      <th style="color:{SLATE};text-align:left;padding:10px 16px;font-size:11px;letter-spacing:1px;font-weight:600;">STRATEGY</th>
      <th style="color:{SLATE};text-align:center;font-size:11px;letter-spacing:1px;font-weight:600;">MEDIAN CAGR</th>
      <th style="color:{SLATE};text-align:center;font-size:11px;letter-spacing:1px;font-weight:600;">EXPECTED RANGE (P25–P75)</th>
      <th style="color:{SLATE};text-align:center;font-size:11px;letter-spacing:1px;font-weight:600;">WORST MDD</th>
      <th style="color:{SLATE};text-align:center;font-size:11px;letter-spacing:1px;font-weight:600;">SHARPE PROXY</th>
    </tr>
    {rows_html}
  </table>
</div>
"""
display(HTML(banner))

## 📈 The Growth Story
### Rolling CAGR Timeline — Performance Through Historical Market Regimes

Each point on the chart represents the **annualised return (CAGR) of a rolling window** ending on that date.  
Green shading marks periods of **above-median CAGR**; red shading marks **below-median** regimes.

> Tracking rolling CAGR over time reveals how *consistently* a strategy delivers — not just what it returned on average, but how reliable that return was across all historical market regimes: 70s inflation, 80s bull run, Dotcom crash, GFC, and Covid.

In [35]:
fig = go.Figure()

for i, (col, name) in enumerate(zip(cagr_cols, strat_names)):
    y_vals = df[col] * 100
    color  = STRAT_PALETTE[i % len(STRAT_PALETTE)]
    median = y_vals.median()

    # Shaded above/below bands only for the primary strategy
    if i == 0:
        y_above = y_vals.clip(lower=median)
        y_below = y_vals.clip(upper=median)
        fig.add_trace(go.Scatter(
            x=df.index, y=y_above, fill='tozeroy',
            fillcolor='rgba(46,204,113,0.10)', line=dict(width=0),
            showlegend=False, hoverinfo='skip'
        ))
        fig.add_trace(go.Scatter(
            x=df.index, y=y_below, fill='tozeroy',
            fillcolor='rgba(231,76,60,0.10)', line=dict(width=0),
            showlegend=False, hoverinfo='skip'
        ))

    fig.add_trace(go.Scatter(
        x=df.index, y=y_vals, mode='lines', name=name,
        line=dict(color=color, width=2.2),
        hovertemplate=f'<b>{name}</b><br>%{{x|%Y-%m-%d}}<br>CAGR: %{{y:.2f}}%<extra></extra>'
    ))

# Median dotted reference
p_median = df[cagr_cols[0]].median() * 100
fig.add_hline(y=p_median, line_dash='dot', line_color='rgba(255,255,255,0.2)',
              annotation_text=f'Primary median {p_median:.1f}%',
              annotation_position='top left', annotation_font_color=LIGHT)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=NAVY, plot_bgcolor='#121E2A',
    title=dict(text='Rolling CAGR Timeline', font=dict(size=20, color=LIGHT), x=0.02),
    xaxis=dict(title='Window End Date', gridcolor='#1E2F3E', showspikes=True),
    yaxis=dict(title='Annualised CAGR (%)', gridcolor='#1E2F3E',
               zeroline=True, zerolinecolor='rgba(255,255,255,0.25)'),
    legend=dict(bgcolor='rgba(13,27,42,0.9)', bordercolor='#1E2F3E', borderwidth=1),
    hovermode='x unified', height=520, margin=dict(t=60, b=50)
)
fig.show()

> ### 📌 Educational Note: CAGR vs Simple Total Return
>
> **CAGR (Compound Annual Growth Rate)** answers: *"If this portfolio grew steadily every year, what rate would produce the same final balance?"*  
> It removes the distortion of one-off lucky years and is the most honest single-number summary of long-term performance.
>
> **Simple Total Return** is just `(Final Value / Initial Value) − 1`.  
> A strategy that doubles in 40 years has a 100% total return, but a CAGR of only **~1.75%** — barely beating inflation.
>
> *Always anchor your expectations to CAGR, not the headline percentage splashed on marketing materials.*

## 📊 Performance Distributions
### What Is the Realistic Range of Outcomes?

- **Violin** — shows the full probability density; wide = more variance = less predictable.  
- **Box whiskers** — mark the 25th–75th percentile (interquartile range).  
- The **red dashed line** marks the zero-return breakeven.  

A strategy with a narrower distribution is easier to *hold* through bad periods.

In [36]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('CAGR Distribution (Violin)', 'CAGR Quartile Breakdown (Box)'),
    horizontal_spacing=0.08
)

for i, (col, name) in enumerate(zip(cagr_cols, strat_names)):
    color = STRAT_PALETTE[i % len(STRAT_PALETTE)]
    r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    y_vals = df[col] * 100

    fig.add_trace(go.Violin(
        y=y_vals, name=name, box_visible=True,
        meanline_visible=True, points=False,
        line_color=color, fillcolor=f'rgba({r},{g},{b},0.2)',
        showlegend=True
    ), row=1, col=1)

    fig.add_trace(go.Box(
        y=y_vals, name=name, boxmean='sd',
        marker_color=color, line_color=color,
        fillcolor=f'rgba({r},{g},{b},0.25)',
        showlegend=False
    ), row=1, col=2)

for c in [1, 2]:
    fig.add_hline(y=0, line_dash='dash', line_color=CRIMSON,
                  annotation_text='Breakeven' if c == 2 else '',
                  annotation_font_color=CRIMSON, row=1, col=c)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=NAVY, plot_bgcolor='#121E2A',
    title=dict(text='CAGR Distribution Analysis', font=dict(size=20, color=LIGHT), x=0.02),
    yaxis=dict(title='CAGR (%)', gridcolor='#1E2F3E'),
    yaxis2=dict(gridcolor='#1E2F3E'),
    legend=dict(bgcolor='rgba(13,27,42,0.9)', bordercolor='#1E2F3E', borderwidth=1),
    height=520, margin=dict(t=80, b=50)
)
fig.show()

## 🌊 Under Stress — Drawdown Deep Dive
### When Did It Hurt? How Bad Was It?

The **underwater chart** below shows the Max Drawdown experienced by rolling windows ending at each date.  
The crimson fill tracks the primary strategy's loss exposure over time.

Look for two things:
- **Depth** — how far below zero?
- **Duration** — how long did elevated drawdown persist?

In [37]:
fig = go.Figure()

for i, (col, name) in enumerate(zip(mdd_cols, strat_names)):
    y_vals = df[col] * 100
    color  = STRAT_PALETTE[i % len(STRAT_PALETTE)]

    if i == 0:
        fig.add_trace(go.Scatter(
            x=df.index, y=y_vals, fill='tozeroy',
            fillcolor='rgba(231,76,60,0.18)',
            line=dict(color=CRIMSON, width=2), name=name,
            hovertemplate=f'<b>{name}</b><br>%{{x|%Y-%m-%d}}<br>MDD: %{{y:.1f}}%<extra></extra>'
        ))
    else:
        fig.add_trace(go.Scatter(
            x=df.index, y=y_vals, mode='lines', name=name,
            line=dict(color=color, width=1.5, dash='dot'),
            hovertemplate=f'<b>{name}</b><br>MDD: %{{y:.1f}}%<extra></extra>'
        ))

median_mdd = df[mdd_cols[0]].median() * 100
fig.add_hline(y=median_mdd, line_dash='dot', line_color='rgba(255,255,255,0.3)',
              annotation_text=f'Median MDD {median_mdd:.1f}%',
              annotation_position='top right', annotation_font_color=LIGHT)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=NAVY, plot_bgcolor='#121E2A',
    title=dict(text='Underwater Drawdown Chart — Rolling MDD Over Time',
               font=dict(size=20, color=LIGHT), x=0.02),
    xaxis=dict(title='Window End Date', gridcolor='#1E2F3E'),
    yaxis=dict(title='Max Drawdown (%)', gridcolor='#1E2F3E'),
    legend=dict(bgcolor='rgba(13,27,42,0.9)', bordercolor='#1E2F3E', borderwidth=1),
    hovermode='x unified', height=480, margin=dict(t=60, b=50)
)
fig.show()

In [38]:
# ── Pain Period: longest consecutive runs in the worst-25th-percentile MDD regime
mdd_series = df[mdd_cols[0]] * 100
threshold  = mdd_series.quantile(0.25)   # below this = 'severe drawdown'
is_stress  = mdd_series < threshold

stress_periods, in_period, start_date = [], False, None
for date, stressed in is_stress.items():
    if stressed and not in_period:
        in_period, start_date = True, date
    elif not stressed and in_period:
        in_period = False
        stress_periods.append({
            'Start': start_date.date(),
            'End':   date.date(),
            'Duration (days)': (date - start_date).days,
            'Worst MDD (%)':   round(mdd_series.loc[start_date:date].min(), 2)
        })
if in_period:
    stress_periods.append({
        'Start': start_date.date(),
        'End':   df.index[-1].date(),
        'Duration (days)': (df.index[-1] - start_date).days,
        'Worst MDD (%)':   round(mdd_series.loc[start_date:].min(), 2)
    })

pain_df = (
    pd.DataFrame(stress_periods)
    .sort_values('Duration (days)', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
pain_df.index += 1

display(HTML(
    f'<div style="background:{NAVY};border-radius:10px;padding:18px 26px;'
    f'font-family:Segoe UI,Arial,sans-serif;margin:8px 0;">'
    f'<h3 style="color:{LIGHT};margin:0 0 4px;font-size:14px;letter-spacing:2px;">PAIN PERIOD SUMMARY</h3>'
    f'<p style="color:{SLATE};margin:0;font-size:12px;">'
    f'Top 10 longest consecutive stretches with MDD in the worst quartile — {strat_names[0]}</p></div>'
))
display(
    pain_df.style
    .set_properties(**{'background-color': '#121E2A', 'color': LIGHT, 'border-color': '#1E2F3E'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#1A2B3C'), ('color', SLATE), ('font-size', '11px')]
    }])
    .format({'Worst MDD (%)': '{:.2f}%'})
    .bar(subset=['Duration (days)'], color='#E74C3C44', vmin=0)
    .bar(subset=['Worst MDD (%)'], color='#E74C3C88',
         vmin=pain_df['Worst MDD (%)'].min(), vmax=0)
)

,Start,End,Duration (days),Worst MDD (%)
1,2019-12-23,2020-02-07,46,-59.89%
2,2016-03-24,2016-05-09,46,-59.89%
3,2017-06-23,2017-08-07,45,-59.89%
4,2012-06-29,2012-08-13,45,-59.89%
5,2014-12-26,2015-02-09,45,-59.89%
6,2024-12-30,2025-02-11,43,-59.89%
7,2021-03-26,2021-05-07,42,-59.89%
8,2018-09-24,2018-11-05,42,-59.89%
9,2013-09-27,2013-11-08,42,-59.89%
10,2022-07-01,2022-08-12,42,-59.89%


> ### 📌 Educational Note: Why Max Drawdown Is the True Measure of Risk
>
> **CAGR tells you the good news.  Max Drawdown tells you the bad.**
>
> Imagine two strategies that both compound at 10%/year over 20 years:
> - Strategy A never drops more than **15%** peak-to-trough.
> - Strategy B periodically drops **55%** before recovering.
>
> They have identical CAGRs, but Strategy B forces most investors to **sell at the worst possible time** —
> converting a temporary paper loss into a permanent one.  
> This is *behavioural risk*, the primary reason retail investors underperform the funds they invest in.
>
> **The best strategy is the one you can hold through its worst period — not the one with the highest backtest return.**

## 🎯 Risk Profile — CDF Analysis
### What Is the Probability of Experiencing a Given Drawdown?

The **Cumulative Distribution Function (CDF)** of Max Drawdown answers:  
*"What percentage of historical rolling windows experienced a drawdown worse than X%?"*

A steep curve = tight distribution = more predictable downside.  
A flat curve = fat tail = higher probability of extreme losses.

In [39]:
fig = go.Figure()

for i, (col, name) in enumerate(zip(mdd_cols, strat_names)):
    color = STRAT_PALETTE[i % len(STRAT_PALETTE)]
    sorted_mdd = np.sort(df[col] * 100)
    y_cdf = np.arange(1, len(sorted_mdd) + 1) / len(sorted_mdd) * 100

    fig.add_trace(go.Scatter(
        x=sorted_mdd, y=y_cdf, mode='lines', name=name,
        line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{name}</b><br>MDD ≤ %{{x:.1f}}%<br>Probability: %{{y:.1f}}%<extra></extra>'
    ))

for pct, label, clr in [(-20, '−20% Risk Zone', GOLD), (-40, '−40% Danger Zone', CRIMSON)]:
    fig.add_vline(x=pct, line_dash='dash', line_color=clr,
                  annotation_text=label, annotation_position='top right',
                  annotation_font_color=clr, annotation_font_size=11)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=NAVY, plot_bgcolor='#121E2A',
    title=dict(text='CDF of Max Drawdown — Probability of Experiencing a Given Loss',
               font=dict(size=20, color=LIGHT), x=0.02),
    xaxis=dict(title='Max Drawdown (%)', gridcolor='#1E2F3E'),
    yaxis=dict(title='Cumulative Probability (%)', gridcolor='#1E2F3E',
               ticksuffix='%', range=[0, 100]),
    legend=dict(bgcolor='rgba(13,27,42,0.9)', bordercolor='#1E2F3E', borderwidth=1),
    hovermode='x unified', height=500, margin=dict(t=60, b=50)
)
fig.show()

## 🗓️ Monthly Alpha Map
### When Does the Strategy Historically Shine?

Each cell shows the **median rolling CAGR for windows ending in that calendar month and year**.  
Green = above-average CAGR; Red = below-average CAGR.

> Seasonality in rolling returns can reveal structural tailwinds or headwinds from recurring macro cycles —
> for example, whether windows ending during a particular season consistently benefited from a regime-level trend.

In [40]:
PRIMARY = cagr_cols[0]
df_map  = df[[PRIMARY]].copy()
df_map['Year']  = df_map.index.year
df_map['Month'] = df_map.index.month

pivot = (
    df_map.groupby(['Year', 'Month'])[PRIMARY]
    .median()
    .unstack(fill_value=np.nan) * 100
)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot.columns = [month_names[m - 1] for m in pivot.columns]

z_vals = pivot.values
valid  = z_vals[~np.isnan(z_vals)]
zmid   = df[PRIMARY].median() * 100

fig = go.Figure(data=go.Heatmap(
    z=z_vals,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale=[
        [0.0, CRIMSON],
        [0.5, '#1E2F3E'],
        [1.0, GREEN]
    ],
    zmid=zmid,
    zmin=valid.min(), zmax=valid.max(),
    colorbar=dict(
        title=dict(text='CAGR (%)', side='right'),
        tickfont=dict(color=LIGHT),
        outlinecolor=NAVY
    ),
    hovertemplate='<b>%{y} %{x}</b><br>Median CAGR: %{z:.2f}%<extra></extra>'
))

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=NAVY, plot_bgcolor=NAVY,
    title=dict(
        text=f'Monthly Alpha Map — {strat_names[0]}',
        font=dict(size=20, color=LIGHT), x=0.02
    ),
    xaxis=dict(title='Month', tickfont=dict(color=LIGHT)),
    yaxis=dict(title='Year', tickfont=dict(color=LIGHT), autorange='reversed'),
    height=max(420, len(pivot) * 22 + 140),
    margin=dict(t=60, b=60)
)
fig.show()